# Exercise 1: Advanced Agent Patterns with LangGraph

In this exercise, you will learn to:
1. Build a custom agent using LangGraph
2. Define state schemas
3. Create nodes and edges
4. Implement conditional routing
5. Add error handling
6. Implement human-in-the-loop approval

**Prerequisites**: Make sure you have completed Lecture 5 exercises and have your OpenAI API key configured.

## Setup

First, let's install the required packages and set up our environment.

In [ ]:
# Install required packages
!pip install langchain langchain-openai langgraph tenacity --quiet

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

print("Environment variables loaded successfully!")

In [ ]:
from typing import Annotated, TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
import json

# Initialize the LLM
llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="gpt-5.4-mini")

## Part 1: Understanding LangGraph State

In LangGraph, state is a TypedDict that flows through all nodes in your graph. Let's start by defining a simple state.

In [ ]:
# Define the state schema
class SimpleState(TypedDict):
    """State for our simple agent."""
    # The add_messages annotation tells LangGraph to append new messages
    # rather than replacing the entire list
    messages: Annotated[list, add_messages]

# Let's see how add_messages works
from langgraph.graph.message import add_messages

# Simulating state updates
current_messages = [HumanMessage(content="Hello!")]
new_messages = [AIMessage(content="Hi there!")]

# add_messages appends rather than replaces
result = add_messages(current_messages, new_messages)
print("Combined messages:")
for msg in result:
    print(f"  {msg.__class__.__name__}: {msg.content}")

## Part 2: Building Your First LangGraph Agent

Let's build a simple agent that can use tools.

In [ ]:
# Define some tools for our agent
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    # Simulated weather data
    weather_data = {
        "new york": "Sunny, 72°F",
        "london": "Cloudy, 58°F",
        "tokyo": "Rainy, 65°F",
        "paris": "Partly cloudy, 68°F"
    }
    return weather_data.get(location.lower(), f"Weather data not available for {location}")

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Example: '2 + 2' or '10 * 5'"""
    try:
        # Safe evaluation of mathematical expressions
        allowed_chars = set('0123456789+-*/().% ')
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

# List of tools
tools = [get_weather, calculate]

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

print("Tools defined:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

In [ ]:
# Define the agent node - this is where the LLM decides what to do
def agent_node(state: SimpleState) -> dict:
    """The agent node calls the LLM to decide the next action."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Define the routing function
def should_continue(state: SimpleState) -> Literal["tools", "end"]:
    """Determine whether to continue to tools or end."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the LLM made tool calls, route to tools
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    
    # Otherwise, we're done
    return "end"

In [ ]:
# Build the graph
graph = StateGraph(SimpleState)

# Add nodes
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))  # ToolNode is a prebuilt node for executing tools

# Add edges
graph.add_edge(START, "agent")  # Start with the agent
graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",  # If tools, go to tools node
        "end": END  # If end, finish
    }
)
graph.add_edge("tools", "agent")  # After tools, go back to agent

# Compile the graph
app = graph.compile()

print("Graph compiled successfully!")

In [ ]:
# Visualize the graph (optional - requires additional dependencies)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Graph visualization not available. Graph structure:")
    print(app.get_graph().draw_ascii())

In [ ]:
# Test the agent!
def run_agent(query: str):
    """Run the agent with a query and display the results."""
    print(f"\n{'='*50}")
    print(f"Query: {query}")
    print('='*50)
    
    result = app.invoke({"messages": [HumanMessage(content=query)]})
    
    print("\nConversation:")
    for msg in result["messages"]:
        if isinstance(msg, HumanMessage):
            print(f"\n👤 Human: {msg.content}")
        elif isinstance(msg, AIMessage):
            if msg.tool_calls:
                print(f"\n🤖 AI: [Calling tools: {[tc['name'] for tc in msg.tool_calls]}]")
            else:
                print(f"\n🤖 AI: {msg.content}")
        elif isinstance(msg, ToolMessage):
            print(f"\n🔧 Tool ({msg.name}): {msg.content}")
    
    return result

# Test with different queries
run_agent("What's the weather in Tokyo?")

In [ ]:
# Test with a calculation
run_agent("What is 15% of 230?")

In [ ]:
# Test with multiple tools
run_agent("What's the weather in London? Also, calculate 72 - 58.")

## Part 3: Adding More Complex State

Let's enhance our state to track more information.

In [ ]:
# Enhanced state with additional tracking
class EnhancedState(TypedDict):
    messages: Annotated[list, add_messages]
    iteration_count: int
    tools_used: list[str]
    status: str

def enhanced_agent_node(state: EnhancedState) -> dict:
    """Enhanced agent that tracks iterations."""
    messages = state["messages"]
    iteration = state.get("iteration_count", 0) + 1
    
    # Add system message about iteration
    if iteration > 5:
        # Prevent infinite loops
        return {
            "messages": [AIMessage(content="I've reached the maximum number of iterations. Here's what I found so far.")],
            "status": "max_iterations_reached",
            "iteration_count": iteration
        }
    
    response = llm_with_tools.invoke(messages)
    
    # Track which tools were called
    tools_used = state.get("tools_used", [])
    if hasattr(response, 'tool_calls') and response.tool_calls:
        tools_used = tools_used + [tc['name'] for tc in response.tool_calls]
    
    return {
        "messages": [response],
        "iteration_count": iteration,
        "tools_used": tools_used,
        "status": "in_progress"
    }

def enhanced_should_continue(state: EnhancedState) -> Literal["tools", "end"]:
    """Check if we should continue or end."""
    # Check for max iterations
    if state.get("status") == "max_iterations_reached":
        return "end"
    
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    
    return "end"

# Build enhanced graph
enhanced_graph = StateGraph(EnhancedState)
enhanced_graph.add_node("agent", enhanced_agent_node)
enhanced_graph.add_node("tools", ToolNode(tools))
enhanced_graph.add_edge(START, "agent")
enhanced_graph.add_conditional_edges("agent", enhanced_should_continue, {"tools": "tools", "end": END})
enhanced_graph.add_edge("tools", "agent")

enhanced_app = enhanced_graph.compile()
print("Enhanced graph compiled!")

In [ ]:
# Test enhanced agent
result = enhanced_app.invoke({
    "messages": [HumanMessage(content="What's the weather in Paris and New York? Then calculate the temperature difference.")],
    "iteration_count": 0,
    "tools_used": [],
    "status": "starting"
})

print(f"\n📊 Agent Statistics:")
print(f"   Iterations: {result['iteration_count']}")
print(f"   Tools used: {result['tools_used']}")
print(f"   Final status: {result['status']}")
print(f"\n🤖 Final response: {result['messages'][-1].content}")

## Part 4: Error Handling with Retry Logic

Let's add error handling using the `tenacity` library.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import random

# Simulate an unreliable API
@tool
def unreliable_search(query: str) -> str:
    """Search for information. This API is sometimes unreliable."""
    # Simulate random failures (30% failure rate)
    if random.random() < 0.3:
        raise Exception("API temporarily unavailable")
    return f"Search results for '{query}': Found relevant information about {query}."

# Wrap the tool with retry logic
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=4),
    retry=retry_if_exception_type(Exception)
)
def search_with_retry(query: str) -> str:
    """Search with automatic retry on failure."""
    return unreliable_search.invoke(query)

# Test the retry logic
print("Testing unreliable search with retry:")
for i in range(5):
    try:
        result = search_with_retry("LangGraph tutorial")
        print(f"  Attempt {i+1}: Success - {result[:50]}...")
    except Exception as e:
        print(f"  Attempt {i+1}: Failed after retries - {e}")

## Part 5: Human-in-the-Loop

Let's add human approval before executing sensitive actions.

In [ ]:
# Define a "sensitive" tool that requires approval
@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient. This is a sensitive action."""
    return f"Email sent to {to} with subject '{subject}'"

# State with approval tracking
class ApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    pending_action: dict | None
    approved: bool

# Tools for this example
approval_tools = [get_weather, send_email]
llm_with_approval_tools = llm.bind_tools(approval_tools)

def approval_agent_node(state: ApprovalState) -> dict:
    """Agent that checks for sensitive actions."""
    messages = state["messages"]
    response = llm_with_approval_tools.invoke(messages)
    
    # Check if the response includes a sensitive tool call
    if hasattr(response, 'tool_calls') and response.tool_calls:
        for tc in response.tool_calls:
            if tc['name'] == 'send_email':
                # This is sensitive - needs approval
                return {
                    "messages": [response],
                    "pending_action": tc,
                    "approved": False
                }
    
    return {
        "messages": [response],
        "pending_action": None,
        "approved": True
    }

def check_approval(state: ApprovalState) -> Literal["tools", "await_approval", "end"]:
    """Route based on whether approval is needed."""
    if state.get("pending_action") and not state.get("approved"):
        return "await_approval"
    
    messages = state["messages"]
    last_message = messages[-1]
    
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    
    return "end"

def approval_node(state: ApprovalState) -> dict:
    """Node that handles approval requests."""
    pending = state["pending_action"]
    print(f"\n⚠️  APPROVAL REQUIRED ⚠️")
    print(f"   Action: {pending['name']}")
    print(f"   Arguments: {json.dumps(pending['args'], indent=2)}")
    
    # In a real application, this would wait for user input
    # For this exercise, we'll simulate approval
    user_input = input("   Approve? (yes/no): ").strip().lower()
    approved = user_input == "yes"
    
    if approved:
        print("   ✅ Approved!")
        return {"approved": True}
    else:
        print("   ❌ Rejected!")
        return {
            "messages": [AIMessage(content="The action was not approved by the user.")],
            "pending_action": None,
            "approved": False
        }

In [ ]:
# Build graph with approval
approval_graph = StateGraph(ApprovalState)

approval_graph.add_node("agent", approval_agent_node)
approval_graph.add_node("tools", ToolNode(approval_tools))
approval_graph.add_node("await_approval", approval_node)

approval_graph.add_edge(START, "agent")
approval_graph.add_conditional_edges(
    "agent",
    check_approval,
    {
        "tools": "tools",
        "await_approval": "await_approval",
        "end": END
    }
)
approval_graph.add_edge("tools", "agent")
approval_graph.add_conditional_edges(
    "await_approval",
    lambda s: "tools" if s.get("approved") else "end",
    {"tools": "tools", "end": END}
)

approval_app = approval_graph.compile()
print("Approval graph compiled!")

In [ ]:
# Test with a non-sensitive action (no approval needed)
print("Test 1: Non-sensitive action")
result = approval_app.invoke({
    "messages": [HumanMessage(content="What's the weather in London?")],
    "pending_action": None,
    "approved": True
})
print(f"\n🤖 Response: {result['messages'][-1].content}")

In [ ]:
# Test with a sensitive action (approval needed)
print("\nTest 2: Sensitive action (type 'yes' or 'no' when prompted)")
result = approval_app.invoke({
    "messages": [HumanMessage(content="Send an email to alice@example.com with subject 'Meeting' and body 'Let's meet tomorrow.'")],
    "pending_action": None,
    "approved": True
})
print(f"\n🤖 Response: {result['messages'][-1].content}")

## 🎯 Challenge Exercises

Now it's your turn! Complete the following challenges:

### Challenge 1: Add a New Tool

Add a `get_time` tool that returns the current time for a given timezone.

In [ ]:
# TODO: Implement the get_time tool
from datetime import datetime

@tool
def get_time(timezone: str) -> str:
    """Get the current time for a timezone. Examples: 'UTC', 'EST', 'PST', 'JST'"""
    # Your implementation here
    pass

# Test your tool
# print(get_time.invoke("UTC"))

### Challenge 2: Add Timeout Handling

Modify the agent to timeout if it takes too long to complete.

In [ ]:
# TODO: Create an agent with timeout handling
# Hint: Track the start time in state and check elapsed time in the routing function

import time

class TimeoutState(TypedDict):
    messages: Annotated[list, add_messages]
    start_time: float
    max_seconds: int

# Your implementation here

### Challenge 3: Multi-Step Approval

Create an agent that requires different approval levels for different actions (e.g., low-risk, medium-risk, high-risk).

In [ ]:
# TODO: Implement multi-level approval
# Define tools with different risk levels
# Route to different approval nodes based on risk

# Your implementation here

## Summary

In this exercise, you learned:

1. **State Management**: How to define and use TypedDict state schemas with annotations
2. **Graph Building**: Creating nodes, edges, and conditional routing
3. **Tool Integration**: Binding tools to LLMs and executing them
4. **Error Handling**: Using retry logic for unreliable operations
5. **Human-in-the-Loop**: Adding approval workflows for sensitive actions

These patterns form the foundation for building robust, production-ready agents!